# submit_d128bag_s7s99e18_55_w440_p205\n5m plus multi-seed daily d128 output fusion.\n

In [ ]:

import os
import gc
import json
import time
import logging

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import dai

logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s", datefmt="%Y-%m-%d %H:%M:%S")
log = logging.getLogger(__name__)

CHUNK_SIZE = 30
BUFFER_DAYS = 70
PRED_BATCH_SIZE = 1024

PRICE_COLS = ["open", "high", "low", "close"]
TRADE_COLS = ["volume", "amount", "deal_number"]
ASK_PRICE = ["ask_price1", "ask_price2", "ask_price3"]
BID_PRICE = ["bid_price1", "bid_price2", "bid_price3"]
ASK_VOL = ["ask_volume1", "ask_volume2", "ask_volume3"]
BID_VOL = ["bid_volume1", "bid_volume2", "bid_volume3"]
RAW_COLS = PRICE_COLS + TRADE_COLS + ASK_PRICE + BID_PRICE + ASK_VOL + BID_VOL
LOG1P_COLS = ["volume", "amount", "deal_number", "ask_volume1", "ask_volume2", "ask_volume3", "bid_volume1", "bid_volume2", "bid_volume3"]


class PatchEmbedding(nn.Module):
    def __init__(self, n_feat, patch_size, d_model):
        super().__init__()
        self.patch_size = patch_size
        self.proj = nn.Linear(n_feat * patch_size, d_model)

    def forward(self, x):
        b, l, c = x.shape
        n_patch = l // self.patch_size
        x = x[:, : n_patch * self.patch_size, :]
        x = x.reshape(b, n_patch, self.patch_size * c)
        return self.proj(x)


class StockTransformer(nn.Module):
    def __init__(self, n_feat=19, patch_size=12, d_model=256, nhead=8, nlayers=4, dim_ff=512, dropout=0.1, seq_len=240):
        super().__init__()
        n_patch = seq_len // patch_size
        self.patch_embed = PatchEmbedding(n_feat, patch_size, d_model)
        self.pos = nn.Parameter(torch.zeros(1, n_patch, d_model))
        layer = nn.TransformerEncoderLayer(
            d_model, nhead, dim_ff, dropout,
            batch_first=True, activation="gelu", norm_first=True,
        )
        self.encoder = nn.TransformerEncoder(layer, nlayers, enable_nested_tensor=False)
        self.head = nn.Sequential(nn.LayerNorm(d_model), nn.Dropout(dropout), nn.Linear(d_model, 1))

    def forward(self, x):
        h = self.patch_embed(x) + self.pos
        h = self.encoder(h).mean(dim=1)
        return self.head(h).squeeze(-1)


def resolve_model_path(path):
    candidates = [path, os.path.join(os.getcwd(), path)]
    try:
        candidates.append(os.path.join(os.path.dirname(__file__), path))
    except NameError:
        pass
    for p in candidates:
        if p and os.path.exists(p):
            return p
    raise FileNotFoundError("model not found: %s" % path)


def load_predictor(path, device):
    with open(resolve_model_path(path), "r", encoding="utf-8") as f:
        payload = json.load(f)
    cfg = dict(payload["model_cfg"])
    cfg = {k: cfg[k] for k in ["n_feat", "patch_size", "d_model", "nhead", "nlayers", "dim_ff", "dropout", "seq_len"] if k in cfg}
    model = StockTransformer(**cfg).to(device)
    state = {}
    for k, meta in payload["state_dict"].items():
        arr = np.asarray(meta["data"], dtype=np.float32).reshape(meta["shape"])
        state[k] = torch.from_numpy(arr)
    model.load_state_dict(state)
    model.eval()
    predictor = {
        "path": path,
        "model": model,
        "freq": payload.get("freq", "bar5m"),
        "seq_len": int(payload.get("seq_len", cfg.get("seq_len", 240))),
        "feature_cols": list(payload.get("feature_cols", RAW_COLS)),
        "mean": np.asarray(payload["mean"], dtype=np.float32),
        "std": np.asarray(payload["std"], dtype=np.float32),
    }
    del payload, state
    gc.collect()
    return predictor


def _chunks(seq, size):
    for i in range(0, len(seq), size):
        yield seq[i : i + size]


def _safe_end_for_filter(end_date):
    ts = pd.Timestamp(end_date)
    if isinstance(end_date, str) and len(end_date.strip()) <= 10:
        ts = ts + pd.Timedelta(days=1) - pd.Timedelta(seconds=1)
    return ts.strftime("%Y-%m-%d %H:%M:%S")


def load_stock_pool(start_date, end_date):
    stk = dai.query(
        "SELECT date, instrument FROM bigalpha_2026_instruments",
        filters={"date": [pd.Timestamp(start_date).strftime("%Y-%m-%d %H:%M:%S"), _safe_end_for_filter(end_date)]},
        compression=True,
    ).df()
    stk = stk.copy()
    stk["date"] = pd.to_datetime(stk["date"]).dt.normalize()
    stk["instrument"] = stk["instrument"].astype(str)
    stk = stk.drop_duplicates(["date", "instrument"]).reset_index(drop=True)
    return stk


def query_bar1m_chunk(table, buf_start, end_date, instruments):
    cols = ["date", "instrument"] + RAW_COLS
    sql = "SELECT %s FROM %s ORDER BY instrument, date" % (", ".join(cols), table)
    return dai.query(
        sql,
        filters={
            "date": [buf_start, _safe_end_for_filter(end_date)],
            "instrument": list(instruments),
        },
        compression=True,
    ).df()


def to_canonical(df):
    df = df.copy()
    if len(df) == 0:
        return pd.DataFrame(columns=["date", "instrument"] + RAW_COLS)
    if not pd.api.types.is_datetime64_any_dtype(df["date"]):
        df["date"] = pd.to_datetime(df["date"])
    if "num_trades" in df.columns and "deal_number" not in df.columns:
        df = df.rename(columns={"num_trades": "deal_number"})
    if "instrument_id" in df.columns and "instrument" not in df.columns:
        df["instrument"] = df["instrument_id"].astype(str)
    for c in RAW_COLS:
        if c not in df.columns:
            df[c] = np.nan
        df[c] = pd.to_numeric(df[c], errors="coerce").astype("float32")
    df["instrument"] = df["instrument"].astype(str)
    return df[["date", "instrument"] + RAW_COLS].reset_index(drop=True)


def preprocess_raw(df):
    df = df.copy()
    for c in LOG1P_COLS:
        if c in df.columns:
            df[c] = np.log1p(df[c].clip(lower=0))
    df[RAW_COLS] = df[RAW_COLS].fillna(0.0)
    return df


def resample_from_1m(df_raw, freq):
    df = to_canonical(df_raw)
    if freq == "bar1m":
        return preprocess_raw(df)

    rule = "5min" if freq == "bar5m" else "30min"
    df = df.sort_values(["instrument", "date"]).reset_index(drop=True)
    df["bar_time"] = df["date"].dt.ceil(rule)
    agg = {
        "open": "first",
        "high": "max",
        "low": "min",
        "close": "last",
        "volume": "last",
        "amount": "last",
        "deal_number": "last",
    }
    for c in ASK_PRICE + BID_PRICE + ASK_VOL + BID_VOL:
        agg[c] = "last"
    out = (
        df.groupby(["instrument", "bar_time"], sort=False, observed=True)
        .agg(agg)
        .reset_index()
        .rename(columns={"bar_time": "date"})
    )
    out = out.dropna(subset=["close"])
    return preprocess_raw(out[["date", "instrument"] + RAW_COLS])


def make_windows(df_feat, feature_cols, seq_len, start_date, end_date):
    sd = pd.Timestamp(start_date).normalize()
    ed = pd.Timestamp(end_date).normalize()
    wins, keys = [], []
    if df_feat is None or len(df_feat) == 0:
        return None, None
    for ins, sub in df_feat.groupby("instrument", sort=False):
        if len(sub) < seq_len:
            continue
        sub = sub.sort_values("date").reset_index(drop=True)
        feats = sub[feature_cols].to_numpy(np.float32)
        day_arr = sub["date"].dt.normalize().to_numpy()
        last_bar = np.flatnonzero(np.append(day_arr[1:] != day_arr[:-1], True))
        dates_by_day = day_arr[last_bar]
        for _, p in enumerate(last_bar):
            d = pd.Timestamp(dates_by_day[_])
            if p + 1 < seq_len or d < sd or d > ed:
                continue
            wins.append(feats[p - seq_len + 1 : p + 1])
            keys.append((d, ins))
    if not keys:
        return None, None
    return np.stack(wins).astype(np.float32), pd.DataFrame(keys, columns=["date", "instrument"])


def predict_array(predictor, x, device):
    x = ((x - predictor["mean"]) / np.where(predictor["std"] > 1e-8, predictor["std"], 1.0)).astype(np.float32)
    xt = torch.from_numpy(x)
    preds = []
    with torch.no_grad():
        for i in range(0, len(xt), PRED_BATCH_SIZE):
            xb = xt[i : i + PRED_BATCH_SIZE].to(device)
            preds.append(predictor["model"](xb).detach().cpu().numpy())
    return np.concatenate(preds).astype(np.float64)


def zscore_by_date(idx_df, values):
    s = pd.Series(values, index=idx_df.index, dtype="float64")
    mean = s.groupby(idx_df["date"]).transform("mean")
    std = s.groupby(idx_df["date"]).transform("std").replace(0.0, np.nan)
    z = (s - mean) / std
    return z.replace([np.inf, -np.inf], np.nan).fillna(0.0).to_numpy(np.float64)


def _freq_from_model_path(path):
    if "bar1m" in path:
        return "bar1m"
    if "bar30m" in path:
        return "bar30m"
    return "bar5m"

MODEL_SPECS = [{'path': 'transformer_model_5m.json', 'weight': 1.0, 'freq': 'bar5m'}]


def main_5m(datasources, start_date, end_date):
    t0 = time.time()
    if "bar5m" in datasources:
        table = datasources["bar5m"]
        source_freq = "bar5m"
    elif "bar1m" in datasources:
        table = datasources["bar1m"]
        source_freq = "bar1m"
    else:
        raise KeyError("datasources['bar5m'] or datasources['bar1m'] is required")

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    log.info("prefer-bar5m chunked inference start device=%s source_freq=%s table=%s start=%s end=%s", device, source_freq, table, start_date, end_date)

    predictors = []
    for i, spec in enumerate(MODEL_SPECS):
        path, weight, freq = spec["path"], float(spec["weight"]), spec.get("freq")
        pred = load_predictor(path, device)
        pred["freq"] = freq or _freq_from_model_path(path)
        pred["weight"] = weight
        pred["col"] = "pred_%02d" % i
        predictors.append(pred)
        log.info("loaded model %s freq=%s seq_len=%d weight=%.4f", path, pred["freq"], pred["seq_len"], weight)

    stk_full = load_stock_pool(start_date, end_date)
    instruments = sorted(stk_full["instrument"].unique().tolist())
    log.info("stock pool rows=%d days=%d instruments=%d", len(stk_full), stk_full["date"].nunique(), len(instruments))
    if not instruments:
        raise RuntimeError("empty stock pool")

    buf_start = (pd.Timestamp(start_date) - pd.Timedelta(days=BUFFER_DAYS)).strftime("%Y-%m-%d %H:%M:%S")
    parts = []
    required_freqs = sorted(set(p["freq"] for p in predictors))

    for ci, chunk in enumerate(_chunks(instruments, CHUNK_SIZE), 1):
        log.info("chunk %d instruments=%d", ci, len(chunk))
        df_raw = query_bar1m_chunk(table, buf_start, end_date, chunk)
        if len(df_raw) == 0:
            log.warning("chunk %d empty", ci)
            continue

        freq_frames = {}
        for freq in required_freqs:
            if source_freq == freq:
                freq_frames[freq] = preprocess_raw(to_canonical(df_raw))
            else:
                freq_frames[freq] = resample_from_1m(df_raw, freq)

        window_cache = {}
        chunk_df = None
        for pred in predictors:
            key = (pred["freq"], pred["seq_len"], tuple(pred["feature_cols"]))
            if key not in window_cache:
                x, idx = make_windows(freq_frames[pred["freq"]], pred["feature_cols"], pred["seq_len"], start_date, end_date)
                window_cache[key] = (x, idx)
            x, idx = window_cache[key]
            if x is None or idx is None:
                continue
            vals = predict_array(pred, x, device)
            one = idx.copy()
            one["date"] = pd.to_datetime(one["date"]).dt.normalize()
            one["instrument"] = one["instrument"].astype(str)
            one[pred["col"]] = vals
            if chunk_df is None:
                chunk_df = one
            else:
                chunk_df = chunk_df.merge(one, on=["date", "instrument"], how="outer")

        if chunk_df is not None and len(chunk_df):
            parts.append(chunk_df)

        del df_raw, freq_frames, window_cache, chunk_df
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    if parts:
        pred_df = pd.concat(parts, ignore_index=True)
        pred_df = pred_df.drop_duplicates(["date", "instrument"], keep="last")
    else:
        pred_df = stk_full.copy()

    if parts:
        final = np.zeros(len(pred_df), dtype=np.float64)
        for pred in predictors:
            col = pred["col"]
            if col in pred_df.columns:
                final += pred["weight"] * zscore_by_date(pred_df, pred_df[col])
        pred_df["score"] = zscore_by_date(pred_df, final)
        pred_df = pred_df[["date", "instrument", "score"]]
    else:
        pred_df["score"] = 0.0
        pred_df = pred_df[["date", "instrument", "score"]]

    result = stk_full.merge(pred_df, on=["date", "instrument"], how="left")
    result["score"] = pd.to_numeric(result["score"], errors="coerce").replace([np.inf, -np.inf], np.nan).fillna(0.0)
    result = result[["date", "instrument", "score"]].drop_duplicates(["date", "instrument"]).reset_index(drop=True)
    log.info("done rows=%d days=%d instruments=%d elapsed=%.1fs", len(result), result["date"].nunique(), result["instrument"].nunique(), time.time() - t0)
    return result


main_5m_ref = main_5m

"""Self-contained inference for the raw daily OHLCV cross-sectional model."""

import gc
import json
import logging
import os
import time

import numpy as np
import pandas as pd
import torch
import torch.nn as nn

try:
    import dai
except ImportError:  # local smoke tests do not need the platform SDK
    dai = None


SCRIPT_DIR = os.path.dirname(os.path.abspath(__file__)) if "__file__" in globals() else os.getcwd()
DAILY_MODEL_PATH = os.path.join(SCRIPT_DIR, "transformer_model_daily.json")

DAILY_PRICE_COLS = ["open", "high", "low", "close"]
DAILY_FLOW_COLS = ["volume", "amount", "deal_number"]
DAILY_COLS = DAILY_PRICE_COLS + DAILY_FLOW_COLS
DAILY_LOG1P_COLS = DAILY_FLOW_COLS

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
)
log = logging.getLogger(__name__)


def _safe_end_for_filter(end_date):
    ts = pd.Timestamp(end_date)
    if isinstance(end_date, str) and len(end_date.strip()) <= 10:
        ts = ts + pd.Timedelta(days=1) - pd.Timedelta(seconds=1)
    return ts.strftime("%Y-%m-%d %H:%M:%S")


def _resolve_model_path(path: str) -> str:
    candidates = [path, os.path.join(os.getcwd(), path), os.path.join(SCRIPT_DIR, path)]
    for candidate in candidates:
        if candidate and os.path.exists(candidate):
            return candidate
    raise FileNotFoundError(path)


class TemporalEncoder(nn.Module):
    def __init__(self, n_feat: int, seq_len: int, d_model: int, nhead: int, nlayers: int, dim_ff: int, dropout: float, revin: int):
        super().__init__()
        self.revin = bool(revin)
        self.proj = nn.Linear(n_feat, d_model)
        self.pos = nn.Parameter(torch.zeros(1, seq_len, d_model))
        self.dw3 = nn.Conv1d(d_model, d_model, kernel_size=3, padding=1, groups=d_model)
        self.dw7 = nn.Conv1d(d_model, d_model, kernel_size=7, padding=3, groups=d_model)
        self.conv_scale = nn.Parameter(torch.tensor(0.1, dtype=torch.float32))
        layer = nn.TransformerEncoderLayer(
            d_model,
            nhead,
            dim_ff,
            dropout,
            batch_first=True,
            activation="gelu",
            norm_first=True,
        )
        self.encoder = nn.TransformerEncoder(layer, nlayers, enable_nested_tensor=False)
        self.pool_q = nn.Parameter(torch.zeros(1, 1, d_model))
        nn.init.normal_(self.pool_q, std=0.02)
        self.pool_attn = nn.MultiheadAttention(d_model, nhead, dropout=dropout, batch_first=True)
        self.norm = nn.LayerNorm(d_model)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        if self.revin:
            mu = x.mean(dim=1, keepdim=True)
            sd = x.std(dim=1, keepdim=True).clamp(min=1e-5)
            x = (x - mu) / sd
        h = self.proj(x) + self.pos[:, : x.shape[1]]
        hc = h.transpose(1, 2)
        h = h + self.conv_scale * (self.dw3(hc) + self.dw7(hc)).transpose(1, 2)
        h = self.encoder(h)
        q = self.pool_q.expand(h.shape[0], -1, -1)
        pooled = self.pool_attn(q, h, h, need_weights=False)[0].squeeze(1)
        return self.norm(pooled)


class DailyOhlcvCrossTransformer(nn.Module):
    def __init__(
        self,
        n_feat: int,
        seq_len: int,
        d_model: int,
        nhead: int,
        ts_layers: int,
        cs_layers: int,
        dim_ff: int,
        dropout: float,
        revin: int = 1,
    ):
        super().__init__()
        self.temporal = TemporalEncoder(n_feat, seq_len, d_model, nhead, ts_layers, dim_ff, dropout, revin)
        cs_layer = nn.TransformerEncoderLayer(
            d_model,
            nhead,
            dim_ff,
            dropout,
            batch_first=True,
            activation="gelu",
            norm_first=True,
        )
        self.cs_encoder = nn.TransformerEncoder(cs_layer, cs_layers, enable_nested_tensor=False)
        self.cs_norm = nn.LayerNorm(d_model)
        self.head = nn.Sequential(nn.LayerNorm(d_model), nn.Dropout(dropout), nn.Linear(d_model, 1))

    def forward(self, x: torch.Tensor, mask: torch.Tensor | None = None) -> torch.Tensor:
        b, n, l, c = x.shape
        z = self.temporal(x.reshape(b * n, l, c)).reshape(b, n, -1)
        if mask is None:
            mask = torch.ones(b, n, dtype=torch.bool, device=x.device)
        z = self.cs_encoder(z, src_key_padding_mask=~mask)
        z = self.cs_norm(z)
        return self.head(z).squeeze(-1)


def _load_model(path: str, device: torch.device):
    with open(_resolve_model_path(path), "r", encoding="utf-8") as f:
        payload = json.load(f)
    state = {}
    for k, meta in payload["state_dict"].items():
        arr = np.asarray(meta["data"], dtype=np.float32).reshape(meta["shape"])
        state[k] = torch.from_numpy(arr)
    model = DailyOhlcvCrossTransformer(**payload["model_cfg"]).to(device)
    model.load_state_dict(state)
    model.eval()
    return {
        "model": model,
        "feature_cols": list(payload.get("feature_cols", DAILY_COLS)),
        "mean": np.asarray(payload["mean"], dtype=np.float32),
        "std": np.asarray(payload["std"], dtype=np.float32),
        "seq_len": int(payload.get("seq_len", payload["model_cfg"]["seq_len"])),
        "freq": str(payload.get("freq", "bar5m")),
    }


def _query_daily_bars(table: str, start_s: str, end_s: str) -> pd.DataFrame:
    last_exc = None
    for deal_col in ("deal_number", "num_trades"):
        sql = f"""
        WITH daily AS (
            SELECT
                strftime(date, '%Y-%m-%d') AS trading_day,
                instrument,
                first(open ORDER BY date) AS open,
                max(high) AS high,
                min(low) AS low,
                last(close ORDER BY date) AS close,
                sum(volume) AS volume,
                sum(amount) AS amount,
                sum({deal_col}) AS deal_number
            FROM {table}
            WHERE date >= TIMESTAMP '{start_s}'
              AND date <= TIMESTAMP '{end_s}'
              AND close > 0
            GROUP BY instrument, strftime(date, '%Y-%m-%d')
        )
        SELECT
            CAST(trading_day AS DATETIME) AS date,
            instrument,
            open, high, low, close, volume, amount, deal_number
        FROM daily
        ORDER BY instrument, date
        """
        try:
            return dai.query(sql, filters={"date": [start_s, end_s]}, compression=True).df()
        except Exception as exc:
            last_exc = exc
    raise last_exc


def _preprocess_daily(df: pd.DataFrame, feature_cols: list[str]) -> pd.DataFrame:
    out = df.copy()
    if len(out) == 0:
        return pd.DataFrame(columns=["date", "instrument"] + feature_cols)
    out["date"] = pd.to_datetime(out["date"]).dt.normalize()
    out["instrument"] = out["instrument"].astype(str)
    out = out.sort_values(["instrument", "date"]).reset_index(drop=True)
    for c in feature_cols:
        if c not in out.columns:
            out[c] = np.nan
        out[c] = pd.to_numeric(out[c], errors="coerce").astype("float32")
    for c in DAILY_PRICE_COLS:
        if c in out.columns:
            out[c] = out.groupby("instrument", sort=False)[c].ffill()
    for c in DAILY_LOG1P_COLS:
        if c in out.columns:
            out[c] = np.log1p(out[c].clip(lower=0))
    out[feature_cols] = out[feature_cols].fillna(0.0).astype("float32")
    return out[["date", "instrument"] + feature_cols]


def _build_windows(df_feat: pd.DataFrame, feature_cols: list[str], seq_len: int, start_date, end_date):
    sd = pd.Timestamp(start_date).normalize()
    ed = pd.Timestamp(end_date).normalize()
    wins = []
    keys = []
    for ins, sub in df_feat.groupby("instrument", sort=False):
        if len(sub) < seq_len:
            continue
        sub = sub.sort_values("date").reset_index(drop=True)
        feats = sub[feature_cols].to_numpy(np.float32)
        dates = pd.to_datetime(sub["date"]).to_numpy()
        for i in range(seq_len - 1, len(sub)):
            d = pd.Timestamp(dates[i])
            if d < sd or d > ed:
                continue
            win = feats[i - seq_len + 1 : i + 1]
            if np.isfinite(win).all():
                wins.append(win)
                keys.append((d, ins))
    if not keys:
        return np.empty((0, seq_len, len(feature_cols)), dtype=np.float32), pd.DataFrame(columns=["date", "instrument"])
    return np.stack(wins, axis=0).astype(np.float32), pd.DataFrame(keys, columns=["date", "instrument"])


def _collate_windows_by_day(x: np.ndarray, idx: pd.DataFrame, days: list[pd.Timestamp]):
    groups = idx.groupby("date", sort=False).indices
    selected = [groups[d] for d in days if d in groups]
    if not selected:
        return None, None, None
    max_n = max(len(ix) for ix in selected)
    seq_len = x.shape[1]
    n_feat = x.shape[2]
    x_pad = np.zeros((len(selected), max_n, seq_len, n_feat), dtype=np.float32)
    mask = np.zeros((len(selected), max_n), dtype=bool)
    out_ix = []
    for i, ix in enumerate(selected):
        n = len(ix)
        x_pad[i, :n] = x[ix]
        mask[i, :n] = True
        out_ix.append(ix)
    return x_pad, mask, out_ix


@torch.no_grad()
def _predict(model: nn.Module, x: np.ndarray, idx: pd.DataFrame, device: torch.device, batch_days: int = 4) -> np.ndarray:
    pred = np.full(len(x), np.nan, dtype=np.float64)
    days = list(pd.Index(idx["date"].unique()).sort_values())
    for i in range(0, len(days), batch_days):
        batch = days[i : i + batch_days]
        x_pad, mask, out_ix = _collate_windows_by_day(x, idx, batch)
        if x_pad is None:
            continue
        xb = torch.from_numpy(x_pad).to(device)
        mb = torch.from_numpy(mask).to(device)
        score = model(xb, mask=mb).float().cpu().numpy()
        for j, ix in enumerate(out_ix):
            pred[ix] = score[j, : len(ix)]
    return pred


def _zscore_by_date(df: pd.DataFrame, col: str) -> np.ndarray:
    s = pd.to_numeric(df[col], errors="coerce").astype("float64")
    g = s.groupby(df["date"])
    z = (s - g.transform("mean")) / g.transform("std").replace(0.0, np.nan)
    return z.replace([np.inf, -np.inf], np.nan).fillna(0.0).to_numpy(np.float64)


def _load_stock_pool(start_date, end_date) -> pd.DataFrame:
    pool = dai.query(
        "SELECT date, instrument FROM bigalpha_2026_instruments",
        filters={"date": [start_date, _safe_end_for_filter(end_date)]},
        compression=True,
    ).df()
    pool = pool.copy()
    pool["date"] = pd.to_datetime(pool["date"]).dt.normalize()
    pool["instrument"] = pool["instrument"].astype(str)
    return pool.drop_duplicates(["date", "instrument"]).reset_index(drop=True)


def main_daily(datasources, start_date, end_date):
    if dai is None:
        raise ImportError("dai is required in the BigQuant runtime")
    t0 = time.time()
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    ckpt = _load_model(DAILY_MODEL_PATH, device)
    model = ckpt["model"]
    feature_cols = ckpt["feature_cols"]
    seq_len = ckpt["seq_len"]
    mean = ckpt["mean"]
    std = ckpt["std"]
    freq = ckpt["freq"]

    table = datasources.get(freq) or datasources.get("bar5m") or datasources.get("bar1m")
    if table is None and len(datasources):
        table = next(iter(datasources.values()))
    if table is None:
        raise KeyError("empty datasources")

    buffer_days = int(seq_len) * 3 + 30
    query_start = (pd.Timestamp(start_date).normalize() - pd.Timedelta(days=buffer_days)).strftime("%Y-%m-%d %H:%M:%S")
    query_end = _safe_end_for_filter(end_date)
    log.info("daily ohlcv inference table=%s start=%s end=%s query_start=%s", table, start_date, end_date, query_start)

    pool = _load_stock_pool(start_date, end_date)
    raw_daily = _query_daily_bars(table, query_start, query_end)
    df_feat = _preprocess_daily(raw_daily, feature_cols)
    x_raw, idx = _build_windows(df_feat, feature_cols, seq_len, start_date, end_date)
    if len(idx):
        x = ((x_raw - mean) / std).astype(np.float32)
        order = idx.sort_values(["date", "instrument"]).index.to_numpy()
        idx = idx.loc[order].reset_index(drop=True)
        x = x[order]
        scores = _predict(model, x, idx, device, batch_days=4)
        pred = idx.copy()
        pred["score"] = scores
        pred["score"] = _zscore_by_date(pred, "score")
    else:
        pred = pd.DataFrame(columns=["date", "instrument", "score"])

    result = pool.merge(pred, on=["date", "instrument"], how="left")
    result["score"] = pd.to_numeric(result["score"], errors="coerce").replace([np.inf, -np.inf], np.nan).fillna(0.0)
    result = result[["date", "instrument", "score"]].drop_duplicates(["date", "instrument"]).reset_index(drop=True)

    del raw_daily, df_feat, x_raw, idx, pred
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    log.info("done rows=%d days=%d elapsed=%.1fs", len(result), result["date"].nunique(), time.time() - t0)
    return result


if __name__ == "__main__":
    log.info("predict_daily_ohlcv_transformer.py defines main(datasources, start_date, end_date)")


DAILY_MODEL_SPECS = [["transformer_model_daily_s7.json", 0.5], ["transformer_model_daily_s99e18.json", 0.5]]


def main_daily_bag(datasources, start_date, end_date):
    if dai is None:
        raise ImportError("dai is required in the BigQuant runtime")
    t0 = time.time()
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    first_ckpt = _load_model(os.path.join(SCRIPT_DIR, DAILY_MODEL_SPECS[0][0]), device)
    feature_cols = first_ckpt["feature_cols"]
    seq_len = first_ckpt["seq_len"]
    freq = first_ckpt["freq"]

    table = datasources.get(freq) or datasources.get("bar5m") or datasources.get("bar1m")
    if table is None and len(datasources):
        table = next(iter(datasources.values()))
    if table is None:
        raise KeyError("empty datasources")

    buffer_days = int(seq_len) * 3 + 30
    query_start = (pd.Timestamp(start_date).normalize() - pd.Timedelta(days=buffer_days)).strftime("%Y-%m-%d %H:%M:%S")
    query_end = _safe_end_for_filter(end_date)
    log.info("daily bag inference models=%d table=%s start=%s end=%s", len(DAILY_MODEL_SPECS), table, start_date, end_date)

    pool = _load_stock_pool(start_date, end_date)
    raw_daily = _query_daily_bars(table, query_start, query_end)
    df_feat = _preprocess_daily(raw_daily, feature_cols)
    x_raw, idx = _build_windows(df_feat, feature_cols, seq_len, start_date, end_date)

    if len(idx):
        order = idx.sort_values(["date", "instrument"]).index.to_numpy()
        idx = idx.loc[order].reset_index(drop=True)
        x_raw = x_raw[order]
        score = np.zeros(len(idx), dtype=np.float64)
        total_w = 0.0
        for model_file, model_weight in DAILY_MODEL_SPECS:
            ckpt = _load_model(os.path.join(SCRIPT_DIR, model_file), device)
            x = ((x_raw - ckpt["mean"]) / ckpt["std"]).astype(np.float32)
            pred_values = _predict(ckpt["model"], x, idx, device, batch_days=4)
            pred_one = idx.copy()
            pred_one["score"] = pred_values
            pred_one["score"] = _zscore_by_date(pred_one, "score")
            w = float(model_weight)
            score += w * pred_one["score"].to_numpy(np.float64)
            total_w += abs(w)
            del ckpt, x, pred_values, pred_one
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
        if total_w <= 1e-12:
            total_w = 1.0
        pred = idx.copy()
        pred["score"] = score / total_w
        pred["score"] = _zscore_by_date(pred, "score")
    else:
        pred = pd.DataFrame(columns=["date", "instrument", "score"])

    result = pool.merge(pred, on=["date", "instrument"], how="left")
    result["score"] = pd.to_numeric(result["score"], errors="coerce").replace([np.inf, -np.inf], np.nan).fillna(0.0)
    result = result[["date", "instrument", "score"]].drop_duplicates(["date", "instrument"]).reset_index(drop=True)

    del first_ckpt, raw_daily, df_feat, x_raw, idx, pred
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    log.info("daily bag done rows=%d days=%d elapsed=%.1fs", len(result), result["date"].nunique(), time.time() - t0)
    return result


FUSION_KIND = "multi_daily_symmetric_softsign"
DUAL_WEIGHT_5M = 0.56000000
DUAL_WEIGHT_DAILY = 0.44000000
SOFTSIGN_PARAM = 2.05000000


def _dual_zscore(frame, col):
    s = pd.to_numeric(frame[col], errors="coerce").astype("float64")
    g = s.groupby(frame["date"])
    z = (s - g.transform("mean")) / g.transform("std").replace(0.0, np.nan)
    return z.replace([np.inf, -np.inf], np.nan).fillna(0.0).to_numpy(np.float64)


def _zscore_values_by_date(frame, values):
    s = pd.Series(np.asarray(values, dtype=np.float64), index=frame.index)
    g = s.groupby(frame["date"])
    z = (s - g.transform("mean")) / g.transform("std").replace(0.0, np.nan)
    return z.replace([np.inf, -np.inf], np.nan).fillna(0.0).to_numpy(np.float64)


def _softsign(z, p):
    p = max(float(p), 1e-6)
    z = np.asarray(z, dtype=np.float64)
    return z / (1.0 + np.abs(z) / p)



def main(datasources, start_date, end_date):
    pred_5m = main_5m(datasources, start_date, end_date).rename(columns={"score": "score_5m"})
    pred_daily = main_daily_bag(datasources, start_date, end_date).rename(columns={"score": "score_daily"})
    out = pred_5m.merge(pred_daily, on=["date", "instrument"], how="left")
    out["score_daily"] = pd.to_numeric(out["score_daily"], errors="coerce").replace([np.inf, -np.inf], np.nan).fillna(0.0)
    z5 = _softsign(_dual_zscore(out, "score_5m"), SOFTSIGN_PARAM)
    zd = _softsign(_dual_zscore(out, "score_daily"), SOFTSIGN_PARAM)
    out["score"] = DUAL_WEIGHT_5M * z5 + DUAL_WEIGHT_DAILY * zd
    out["score"] = _zscore_values_by_date(out, out["score"])
    return out[["date", "instrument", "score"]].drop_duplicates(["date", "instrument"]).reset_index(drop=True)
